# Spicerack - Recipe Recommender
### DS Club Project - Spring 2026

The idea is pretty simple: you tell us what spices you have, we tell you what you can cook.
Also tells you what spice to buy next to unlock the most new recipes.

Dataset: RecipeNLG (~2.2M recipes from Kaggle)

GitHub: https://github.com/laniel123/DS-Club-Project-Showcase-SpiceRack

In [5]:
import pandas as pd
import numpy as np
import os
from joblib import load

---
## Step 1 - Change your inputs here

This is the only cell you need to touch. Edit your pantry and must-use lists, then run all cells.

If you put something in must-use that isnt in your pantry, it gets added automatically.
If a spice isnt recognized it just gets skipped with a warning.

In [2]:
# change these two lists and run everything

# spices you actually have right now
MY_PANTRY = [
    "garlic",
    "cumin",
    "paprika",
    "chili powder",
    "oregano",
    "black pepper",
    "salt",
    "cinnamon",
    "ginger",
]

# spices that MUST show up in every recipe we recommend
# leave as [] if you dont care
MUST_USE = [
    "cumin",
    "garlic",
]

# path to your csv file
CSV_PATH = "cookingdataset/RecipeNLG_dataset.csv"

# how many recipes to load - lower this if its running slow
SAMPLE_SIZE = None  # set to None to load all, or a number like 10000 for a sample

---
## Step 2 - Imports and setup
Run this once, dont touch it

In [3]:
import re
import warnings
import pandas as pd
import numpy as np
import joblib
import time
from sklearn.decomposition import NMF
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import pairwise_distances

# pulling all our spice data from the external file
# spice_data.py needs to be in the same folder as this notebook
from spice_data_v2 import SPICES, ALIASES, CANONICAL_SPICES, FLAVOR_PROFILES, REGION_PROFILES

print(f"loaded {len(CANONICAL_SPICES)} spices, {len(FLAVOR_PROFILES)} flavor profiles, {len(REGION_PROFILES)} regions")

loaded 179 spices, 15 flavor profiles, 16 regions


In [4]:
# helper functions used throughout

from spice_data_v2 import SPICES, ALIASES, CANONICAL_SPICES

def clean(s) -> str:
    # lowercase and strip weird characters
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r"[^a-z\s']", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def to_canonical(spice):
    # normalize a spice name and apply alias mapping
    n = clean(spice)
    return clean(ALIASES.get(n, n))


def validate_spices(raw_list, label):
    # make sure everything the user typed is actually in our vocabulary
    # warns on anything we dont recognize and just skips it
    valid = set()
    bad = []
    for s in raw_list:
        c = to_canonical(s)
        if c in CANONICAL_SPICES:
            valid.add(c)
        else:
            bad.append(s)
    if bad:
        print(f"warning ({label}): didnt recognize these, skipping: {bad}")
    return valid


# build regex patterns for spice extraction from recipe text
# sorted by length so longer matches beat shorter ones (smoked paprika before paprika)
SPICE_PATTERNS = sorted(
    [(sp, clean(sp), re.compile(rf"(^| ){re.escape(clean(sp))}( |$)")) for sp in SPICES],
    key=lambda x: -len(x[0])
)

def get_spices_from_recipe(ingredients):
    # takes a recipe's ingredient list and returns which spices are in it
    if isinstance(ingredients, list):
        raw = " ".join(str(x) for x in ingredients)
    else:
        raw = str(ingredients)
    text = clean(raw)
    found = set()
    for _, norm, pat in SPICE_PATTERNS:
        if pat.search(" " + text + " "):
            found.add(norm)
    return {to_canonical(sp) for sp in found}


def parse_ingredient_string(x) -> list[str]:
    # ingredients are stored as a stringified python list in the csv
    # like '["1 cup flour", "2 eggs"]' so we parse it back
    if isinstance(x, list):
        return [str(i) for i in x]
    if not isinstance(x, str):
        return []
    s = x.strip()
    if s.startswith("[") and s.endswith("]"):
        items = re.findall(r"'([^']*)'|\"([^\"]*)\"", s)
        parsed = [a if a else b for a, b in items]
        return parsed if parsed else [s]
    return [s]



print("helper functions ready")

helper functions ready


---
## Step 3 - Load the data and create the model
Run this once per session. Takes a minute on the full dataset.

In [8]:
#df = pd.read_csv(CSV_PATH)

In [9]:
#df.columns

In [10]:
#df.head()

In [11]:
#sample_df = df.sample(n=1000, random_state=42)

#sample_df = df.sample(n=1000, random_state=42)

# save a smaller sample for quick testing without loading the whole thing every time

#sample_df.columns
#sample_df.to_csv('sample_dataset_1000.csv', index=False)



In [12]:
print(f"loading {CSV_PATH}...")

df = pd.read_csv(CSV_PATH)

if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# parse NER column (clean ingredient tokens) instead of raw ingredients
# NER gives ["vanilla", "flour", "butter"] not "1/2 tsp. vanilla extract"
df["ingredients_parsed"] = df["NER"].apply(parse_ingredient_string)
df["spices"] = df["ingredients_parsed"].apply(get_spices_from_recipe)

# binary matrix: rows = recipes, columns = spices, 1 if recipe contains that spice
mlb = MultiLabelBinarizer(classes=sorted(CANONICAL_SPICES))
X = np.asarray(mlb.fit_transform(df["spices"]))

print(f"done - {len(df):,} recipes loaded")
print(f"matrix shape: {X.shape}")
print(f"avg spices per recipe: {X.sum(axis=1).mean():.1f}")

loading cookingdataset/RecipeNLG_dataset.csv...
done - 2,231,142 recipes loaded
matrix shape: (2231142, 179)
avg spices per recipe: 1.7


In [13]:
test_data = df.copy()

df.head()

,Unnamed: 0,title,ingredients,directions,link,source,NER,ingredients_parsed,spices
0,0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu...","[brown sugar, milk, vanilla, nuts, butter, bit...",{vanilla}
1,1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""beef"", ""chicken breasts"", ""cream of mushroom...","[beef, chicken breasts, cream of mushroom soup...",{}
2,2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""cream cheese"", ""butter"", ""gar...","[frozen corn, cream cheese, butter, garlic pow...","{salt, garlic}"
3,3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken"", ""chicken gravy"", ""cream of mushroo...","[chicken, chicken gravy, cream of mushroom sou...",{}
4,4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""peanut butter"", ""graham cracker crumbs"", ""bu...","[peanut butter, graham cracker crumbs, butter,...",{}


In [14]:
#sample_df.columns

In [15]:
#sample_df.head(15)

In [16]:
print(f"df shape: {df.shape}")
print(f"X shape: {X.shape}")
print(f"spices sample: {list(df['spices'].iloc[0])}")
print(f"non-empty spice rows: {df['spices'].apply(len).gt(0).sum()}")
print(f"avg spices per recipe: {df['spices'].apply(len).mean():.2f}")

df shape: (2231142, 9)
X shape: (2231142, 179)
spices sample: ['vanilla']
non-empty spice rows: 1649782
avg spices per recipe: 1.70


In [17]:
print("first 5 spice sets:")
for i in range(5):
    print(f"  {df['spices'].iloc[i]}")

print()
print("first 5 ingredients_parsed:")
for i in range(5):
    print(f"  {df['ingredients_parsed'].iloc[i][:3]}")

print()
print(f"does df have ingredients_parsed col: {'ingredients_parsed' in df.columns}")
print(f"does df have spices col: {'spices' in df.columns}")
print(f"df columns: {list(df.columns)}")

first 5 spice sets:
  {'vanilla'}
  set()
  {'salt', 'garlic'}
  set()
  set()

first 5 ingredients_parsed:
  ['brown sugar', 'milk', 'vanilla']
  ['beef', 'chicken breasts', 'cream of mushroom soup']
  ['frozen corn', 'cream cheese', 'butter']
  ['chicken', 'chicken gravy', 'cream of mushroom soup']
  ['peanut butter', 'graham cracker crumbs', 'butter']

does df have ingredients_parsed col: True
does df have spices col: True
df columns: ['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source', 'NER', 'ingredients_parsed', 'spices']


In [18]:
from sklearn.cluster       import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics       import silhouette_score

N_CLUSTERS = 100   # bump to 100 for full 2M dataset

# ── cluster ───────────────────────────────────────────────────────────────
# X is (n_recipes, 179) binary matrix from cell 11
# MiniBatchKMeans is the same as KMeans but fast enough for 2M rows

kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS, random_state=42,
    batch_size=4096, n_init=10
)
df["cluster"] = kmeans.fit_predict(X)

# 1. initial clustering
kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS, random_state=42,
    batch_size=4096, n_init=10
)
df["cluster"] = kmeans.fit_predict(X)

# 2. split oversized clusters
SIZE_THRESHOLD = 50_000   # any cluster bigger than this gets split
SUBCLUSTERS    = 5         # how many pieces to break it into
next_id        = N_CLUSTERS  # new sub-cluster IDs start here

for cid in range(N_CLUSTERS):
    mask = df["cluster"] == cid
    if mask.sum() <= SIZE_THRESHOLD:
        continue  # fine as-is, skip

    print(f"splitting cluster {cid} ({mask.sum():,} recipes) into {SUBCLUSTERS}...")
    sub_X      = X[mask.values]
    sub_kmeans = MiniBatchKMeans(
        n_clusters=SUBCLUSTERS, random_state=42,
        batch_size=4096, n_init=5
    )
    sub_labels = sub_kmeans.fit_predict(sub_X)

    # assign new IDs to each sub-cluster
    df.loc[mask, "cluster"] = sub_labels + next_id
    next_id += SUBCLUSTERS

print(f"\ntotal clusters after splitting: {df['cluster'].nunique()}")
print(df["cluster"].value_counts().sort_index().to_string())

print(f"cluster sizes:")
print(df["cluster"].value_counts().sort_index().to_string())

# silhouette score on a 5k sample — tells you how good the clusters are
# >0.2 is reasonable, >0.4 is good
idx = np.random.choice(len(X), min(5000, len(X)), replace=False)
sil = silhouette_score(X[idx], df["cluster"].iloc[idx], metric="jaccard")
print(f"\nsilhouette: {sil:.3f}")

splitting cluster 41 (53,911 recipes) into 5...
splitting cluster 49 (57,286 recipes) into 5...
splitting cluster 63 (582,798 recipes) into 5...
splitting cluster 72 (273,021 recipes) into 5...
splitting cluster 83 (148,669 recipes) into 5...
splitting cluster 84 (111,512 recipes) into 5...

total clusters after splitting: 124
cluster
0       10718
1       16852
2       24343
3        6523
4       13489
5       10300
6        5414
7        4401
8       10749
9        9927
10      12660
11      12127
12      24674
13       6665
14      13454
15      16925
16      10053
17      16477
18       1767
19      14044
20       5987
21      11334
22      11446
23      13830
24      25932
25      13022
26       7294
27      18276
28       9400
29       4081
30      17691
31      43524
32       9838
33      11423
34      17455
35       9635
36       4490
37      18225
38      25957
39      13584
40       4559
42      20656
43      22925
44      10736
45       1151
46       7600
47       9802
48   

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)



silhouette: 0.661


In [19]:
# remove empty clusters from df and remap IDs to be contiguous
df = df[df["cluster"].map(df["cluster"].value_counts()) > 0].reset_index(drop=True)

# remap cluster IDs to 0, 1, 2... with no gaps
old_ids = sorted(df["cluster"].unique())
id_map  = {old: new for new, old in enumerate(old_ids)}
df["cluster"] = df["cluster"].map(id_map)

# also remap cluster_top_spices to match new IDs
if "cluster_top_spices" in globals():
    cluster_top_spices = {id_map[k]: v for k, v in cluster_top_spices.items() if k in id_map}
else:
    cluster_top_spices = {}

print(f"final cluster count: {df['cluster'].nunique()}")
print(f"recipes kept: {len(df):,}")

idx = np.random.choice(len(X), min(5000, len(X)), replace=False)
sil = silhouette_score(X[idx], df["cluster"].iloc[idx], metric="jaccard")
print(f"silhouette after splitting: {sil:.3f}")

final cluster count: 124
recipes kept: 2,231,142


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


silhouette after splitting: 0.669


In [20]:
# what did each cluster find?
# read the top spices per cluster — these describe the flavor group
spice_cols = list(mlb.classes_)
cluster_top_spices = {}

for cid in range(N_CLUSTERS):
    mask    = df["cluster"] == cid
    freq    = X[mask.values].mean(axis=0)
    top_idx = freq.argsort()[::-1][:8]
    top     = [spice_cols[i] for i in top_idx if freq[i] > 0]
    cluster_top_spices[cid] = top
    print(f"cluster {cid:>2}  ({mask.sum():>5} recipes)  {', '.join(top[:6])}")

cluster  0  (10718 recipes)  cinnamon, nutmeg, salt, cloves, garlic, kosher salt
cluster  1  (16852 recipes)  cayenne, rosemary, sage, bay leaf, cardamom, poppy seed
cluster  2  (24343 recipes)  salt, dill, thyme, oregano, cream of tartar, white pepper
cluster  3  ( 6523 recipes)  vanilla, nutmeg, cocoa powder, ginger, mint, cardamom
cluster  4  (13489 recipes)  garlic, cilantro, chili powder, paprika, dill, italian seasoning
cluster  5  (10300 recipes)  salt, vanilla, cream of tartar, kosher salt, cocoa powder, nutmeg
cluster  6  ( 5414 recipes)  oregano, basil, garlic, salt, black pepper, thyme
cluster  7  ( 4401 recipes)  ginger, salt, garlic, cilantro, turmeric, kosher salt
cluster  8  (10749 recipes)  black pepper, salt, mustard, thyme, basil, paprika
cluster  9  ( 9927 recipes)  mustard, salt, parsley, paprika, kosher salt, dill
cluster 10  (12660 recipes)  salt, parsley, garlic, oregano, thyme, bay leaf
cluster 11  (12127 recipes)  garlic, salt, dill, bay leaf, cayenne, italian 

In [47]:
print(df.shape)

(2231142, 10)


In [ ]:

# craete new dataset with cluster labels 

df_sample = df.sample(random_state=42).reset_index(drop=True)

In [48]:
#df.to_csv("/Users/daniellarson/Desktop/SpiceRack/cluster_data.csv", index=False)
#print("saved")

saved


In [41]:
df_sample.columns

Index(['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source',
       'NER', 'ingredients_parsed', 'spices', 'cluster'],
      dtype='object')

In [26]:
N_SVD_DIMS = 100   # bump to 50-100 for full dataset

# compress each recipe from 179 binary dims to 30 dense dims
# after normalize, cosine similarity = dot product (fast)
svd         = TruncatedSVD(n_components=N_SVD_DIMS, random_state=42)
recipe_vecs = svd.fit_transform(X)
recipe_vecs = normalize(recipe_vecs, norm="l2") #this TF-IDF gives each spice a weight based on how rare it is across all recipes. Salt appears in 80% of recipes so it gets a very low weight.

print(f"recipe matrix: {recipe_vecs.shape}")
print(f"variance explained: {svd.explained_variance_ratio_.sum():.1%}")

# save everything the website needs
model = {
    "kmeans":             kmeans,
    "svd":                svd,
    "mlb":                mlb,
    "recipe_matrix":      recipe_vecs,
    "recipe_titles":      df["title"].tolist(),
    "recipe_spices":      [list(s) for s in df["spices"]],
    "cluster_labels":     df["cluster"].tolist(),
    "cluster_top_spices": cluster_top_spices,
    "n_clusters":         N_CLUSTERS,
    "n_recipes":          len(df),
    "silhouette":         round(float(sil), 4),  

}
#joblib.dump(model, "spicerack_model.joblib", compress=3)

#print("model saved.")

print(list(m.keys()))

recipe matrix: (2231142, 100)
variance explained: 100.0%


NameError: name 'm' is not defined

In [ ]:
import joblib
m = joblib.load("models/spicerack_model.joblib")
print("keys:", list(m.keys()))
print("n_recipes:", m["n_recipes"])
print("n_clusters:", m["n_clusters"])
print()

# check a sample recipe
print("recipe_titles sample:", m["recipe_titles"][:3])
print("recipe_spices sample:", m["recipe_spices"][:3])
print("cluster_labels sample:", m["cluster_labels"][:5])
print("recipe_matrix shape:", m["recipe_matrix"].shape)

keys: ['kmeans', 'svd', 'mlb', 'recipe_matrix', 'recipe_titles', 'recipe_spices', 'cluster_labels', 'cluster_top_spices', 'n_clusters', 'n_recipes', 'silhouette']
n_recipes: 2231142
n_clusters: 100

recipe_titles sample: ['No-Bake Nut Cookies', "Jewell Ball'S Chicken", 'Creamy Corn']
recipe_spices sample: [['vanilla'], [], ['salt', 'garlic']]
cluster_labels sample: [118, 108, 97, 108, 108]
recipe_matrix shape: (2231142, 100)


In [ ]:
import numpy as np

cluster_arr = np.array(m["cluster_labels"])
spices_list = m["recipe_spices"]

# check how many recipes have spices in the nearest cluster
test_pantry = {"garlic", "cumin", "paprika", "salt"}
user_bin    = np.asarray(m["mlb"].transform([test_pantry]))
nearest     = int(m["kmeans"].predict(user_bin)[0])

in_cluster  = cluster_arr == nearest
has_spices  = np.array([len(s) > 0 for s in spices_list])

print(f"nearest cluster: {nearest}")
print(f"recipes in cluster: {in_cluster.sum():,}")
print(f"recipes in cluster with spices: {(in_cluster & has_spices).sum():,}")

# try scoring
user_svd = m["svd"].transform(user_bin)[0]
user_svd /= np.linalg.norm(user_svd)
scores   = m["recipe_matrix"] @ user_svd
scores[~in_cluster] = 0
print(f"max score in cluster: {scores.max():.4f}")
print(f"non-zero scores: {(scores > 0).sum():,}")

nearest cluster: 23
recipes in cluster: 13,830
recipes in cluster with spices: 13,830
max score in cluster: 1.0000
non-zero scores: 13,830


---
## Step 4 - Model functions
These dont need to be touched either. Just run them.

In [ ]:
import joblib
import numpy as np

m = joblib.load("spicerack_model.joblib")

# ── step 1: define a test pantry ──────────────────────────────────────────
pantry   = ["garlic", "cumin", "paprika", "salt", "black pepper"]
pantry_set = set(pantry)

# ── step 2: build user vector ─────────────────────────────────────────────
user_bin = np.asarray(m["mlb"].transform([pantry_set]))
user_svd = m["svd"].transform(user_bin)[0]
user_svd = user_svd / np.linalg.norm(user_svd)

print(f"user vector shape: {user_svd.shape}")
print(f"user vector (first 5 values): {user_svd[:5].round(4)}")

# ── step 3: find nearest cluster ──────────────────────────────────────────
nearest = int(m["kmeans"].predict(user_bin)[0])
print(f"\nnearest cluster: {nearest}")
print(f"top spices in that cluster: {m['cluster_top_spices'][nearest]}")

# ── step 4: score every recipe ────────────────────────────────────────────
scores      = m["recipe_matrix"] @ user_svd
cluster_arr = np.array(m["cluster_labels"])
scores[cluster_arr != nearest] = 0

print(f"\nrecipes in cluster: {(cluster_arr == nearest).sum():,}")
print(f"max score: {scores.max():.4f}")
print(f"non-zero scores: {(scores > 0).sum():,}")

# ── step 5: show top 10 ───────────────────────────────────────────────────
top_idx = np.argsort(scores)[::-1][:10]

print(f"\ntop 10 recommendations for pantry {pantry}:")
print(f"{'score':<8} {'title':<45} matched spices")
print("-" * 80)
for i in top_idx:
    if scores[i] <= 0:
        break
    recipe_spices = set(m["recipe_spices"][i])
    matched = sorted(pantry_set & recipe_spices)
    print(f"{scores[i]:.3f}    {m['recipe_titles'][i][:44]:<45} {matched}")

In [ ]:
def get_flavor_profiles(pantry, min_coverage=0.3):
    # check how well the users spices cover each flavor profile
    # only returns profiles where they have at least min_coverage fraction of the spices
    results = []
    for profile, spice_set in FLAVOR_PROFILES.items():
        matched = pantry & spice_set
        coverage = len(matched) / len(spice_set)
        if coverage >= min_coverage:
            results.append((profile, round(coverage, 2), sorted(matched)))
    return sorted(results, key=lambda x: -x[1])


def get_regions(profile_names):
    # maps matched flavor profiles to culinary regions
    results = []
    for region, profiles in REGION_PROFILES.items():
        matched = [p for p in profiles if p in profile_names]
        if matched:
            results.append((region, matched))
    return sorted(results, key=lambda x: -len(x[1]))


def recommend(pantry, must_use, top_k=10, min_match=2):
    # jaccard similarity - compares users spice set to each recipes spice set
    # must_use filter removes any recipe that doesnt have all the required spices
    user_vec = np.asarray(mlb.transform([pantry]))

    if must_use:
        spice_cols = list(mlb.classes_)
        must_idx = [spice_cols.index(s) for s in must_use if s in spice_cols]
        if must_idx:
            must_mask = X[:, must_idx].min(axis=1).astype(bool)
        else:
            must_mask = np.ones(len(df), dtype=bool)
    else:
        must_mask = np.ones(len(df), dtype=bool)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sims = 1.0 - pairwise_distances(user_vec, X, metric="jaccard").flatten()

    match_counts = (X & user_vec).sum(axis=1)
    valid = np.where(must_mask & (match_counts >= min_match))[0]

    if len(valid) == 0:
        print("no recipes matched the must-use filter, showing best results without it")
        valid = np.where(match_counts >= min_match)[0]

    ranked = valid[np.lexsort((-match_counts[valid], -sims[valid]))][:top_k]

    out = df.loc[ranked, ["title"]].copy()
    out["similarity"]     = sims[ranked].round(3)
    out["matched_spices"] = df.loc[ranked, "spices"].apply(lambda s: sorted(s & pantry))
    out["num_matched"]    = match_counts[ranked]
    return out.sort_values(["similarity", "num_matched"], ascending=False).reset_index(drop=True)


def tag_region(recipe_spices):
    # assign a single region label to a recipe based on its spice set
    profiles = get_flavor_profiles(recipe_spices, min_coverage=0.2)
    if not profiles:
        return "Other"
    top_profile = profiles[0][0]
    for region, region_profiles in REGION_PROFILES.items():
        if top_profile in region_profiles:
            return region
    return "Other"


def suggest_next_spice(pantry, top_k=5, min_match=2, threshold=0.4):
    # for every spice you dont have, simulate adding it
    # count how many new recipes become available and rank by that
    baseline = recommend(pantry, must_use=set(), top_k=10_000, min_match=min_match)
    baseline_titles = set(baseline["title"])
    baseline_count  = len(baseline[baseline["similarity"] >= threshold])

    missing = [s for s in CANONICAL_SPICES if s not in pantry]
    print(f"baseline: {baseline_count} recipes above {threshold} similarity")
    print(f"testing {len(missing)} candidate spices...")

    results = []
    for spice in missing:
        expanded = pantry | {spice}
        new_recs = recommend(expanded, must_use=set(), top_k=10_000, min_match=min_match)
        new_count  = len(new_recs[new_recs["similarity"] >= threshold])
        new_titles = set(new_recs["title"]) - baseline_titles
        results.append({
            "spice":          spice,
            "newly_unlocked": new_count - baseline_count,
            "examples":       list(new_titles)[:3],
        })

    return pd.DataFrame(results).sort_values("newly_unlocked", ascending=False).head(top_k).reset_index(drop=True)


print("model functions ready")

model functions ready


italian seasoning in mlb: True
recipes with italian seasoning: 13,357
that is 0.60% of recipes


nearest cluster: 69
cluster top spices: ['marjoram', 'chive', 'mace', 'bay leaf', 'cloves', 'white pepper', 'tarragon', 'sage']
total recipes in cluster: 654
italian seasoning recipes in nearest cluster: 0
italian seasoning recipes across ALL clusters: 13,357


In [38]:
from sklearn.feature_extraction.text import TfidfTransformer

N_SVD_DIMS = 120

# TF-IDF weighting — downweights salt/pepper, upweights rare spices
tfidf   = TfidfTransformer(norm="l2", use_idf=True, smooth_idf=True)
X_tfidf = tfidf.fit_transform(X)

# boost rare spices further by squaring the IDF weights
idf_weights = np.array(tfidf.idf_)
idf_boost   = idf_weights ** 2
idf_boost   = idf_boost / idf_boost.max()
X_boosted   = X_tfidf.multiply(idf_boost)
X_boosted   = normalize(X_boosted, norm="l2")

spice_cols  = list(mlb.classes_)
print(f"salt    weight after boost: {X_boosted[:, spice_cols.index('salt')].mean():.6f}")
print(f"saffron weight after boost: {X_boosted[:, spice_cols.index('saffron')].mean():.6f}")

# SVD compression
svd         = TruncatedSVD(n_components=N_SVD_DIMS, random_state=42)
recipe_vecs = svd.fit_transform(X_boosted)
recipe_vecs = normalize(recipe_vecs, norm="l2")

print(f"recipe matrix: {recipe_vecs.shape}")
print(f"variance explained: {svd.explained_variance_ratio_.sum():.1%}")

# save everything the website needs
model = {
    "kmeans":             kmeans,
    "svd":                svd,
    "mlb":                mlb,
    "tfidf":              tfidf,
    "idf_boost":          idf_boost,
    "recipe_matrix":      recipe_vecs,
    "recipe_titles":      df["title"].tolist(),
    "recipe_spices":      [list(s) for s in df["spices"]],
    "cluster_labels":     df["cluster"].tolist(),
    "cluster_top_spices": cluster_top_spices,
    "n_clusters":         N_CLUSTERS,
    "n_recipes":          len(df),
    "silhouette":         round(float(sil), 4),
}

joblib.dump(model, "spicerack_model2.joblib", compress=3)
print("model saved.")
print("keys:", list(model.keys()))

salt    weight after boost: 0.153267
saffron weight after boost: 0.002192
recipe matrix: (2231142, 120)
variance explained: 100.0%
model saved.
keys: ['kmeans', 'svd', 'mlb', 'tfidf', 'idf_boost', 'recipe_matrix', 'recipe_titles', 'recipe_spices', 'cluster_labels', 'cluster_top_spices', 'n_clusters', 'n_recipes', 'silhouette']


---
## Step 5 - Results
Edit step 1 and re-run just this cell whenever you want to try different spices.

In [ ]:
#old code, old cell. 


# validate the inputs from step 1
"""pantry   = validate_spices(MY_PANTRY, label="pantry")
must_use = validate_spices(MUST_USE,  label="must-use")

extras = must_use - pantry
if extras:
    print(f"added to pantry automatically: {extras}")
    pantry |= extras

print(f"\npantry:   {sorted(pantry)}")
print(f"must-use: {sorted(must_use)}")


# --- flavor profiles ---
print("\n" + "-" * 55)
print("flavor profiles")
print("-" * 55)

profiles = get_flavor_profiles(pantry)
if not profiles:
    print("no strong profiles found - try adding more spices")
else:
    for name, score, matched in profiles:
        bar = "#" * int(score * 10) + "." * (10 - int(score * 10))
        print(f"  [{bar}] {int(score*100)}%  {name}")
        print(f"           spices: {', '.join(matched)}")


# --- culinary regions ---
print("\n" + "-" * 55)
print("cuisines you can cook")
print("-" * 55)

regions = get_regions([p[0] for p in profiles])
if not regions:
    print("no strong regional match found")
else:
    for region, matched_profiles in regions:
        print(f"  {region}")
        print(f"    via: {', '.join(matched_profiles)}")


# --- top recipe recommendations ---
print("\n" + "-" * 55)
if must_use:
    print(f"top recipes (must contain: {sorted(must_use)})")
else:
    print("top recipes")
print("-" * 55)

recs = recommend(pantry, must_use, top_k=10, min_match=2)
print(recs[["title", "similarity", "matched_spices", "num_matched"]].to_string(index=False))


# --- recipes grouped by region ---
print("\n" + "-" * 55)
print("recipes by region")
print("-" * 55)

pool = recommend(pantry, must_use, top_k=200, min_match=2)
pool["region"] = pool["title"].apply(
    lambda t: tag_region(
        df[df["title"] == t]["spices"].iloc[0]
        if (df["title"] == t).any() else set()
    )
)

for region in pool["region"].unique():
    if region == "Other":
        continue
    subset = pool[pool["region"] == region].head(3)
    print(f"\n  {region}")
    for _, row in subset.iterrows():
        print(f"    - {row['title']}  (similarity: {row['similarity']})")


# --- what spice should you buy next ---
print("\n" + "-" * 55)
print("spices to buy next")
print("-" * 55)

next_spice = suggest_next_spice(pantry, top_k=5, min_match=2)
for _, row in next_spice.iterrows():
    print(f"  {row['spice']:<25} {row['newly_unlocked']:+} new recipes")
    if row["examples"]:
        print(f"  {'':25} e.g. {row['examples'][0]}") """ 


pantry:   ['black pepper', 'chili powder', 'cinnamon', 'cumin', 'garlic', 'ginger', 'oregano', 'paprika', 'salt']
must-use: ['cumin', 'garlic']

-------------------------------------------------------
flavor profiles
-------------------------------------------------------
  [###.......] 31%  American BBQ & Southern
           spices: black pepper, chili powder, cumin, garlic, paprika

-------------------------------------------------------
cuisines you can cook
-------------------------------------------------------
  American BBQ / Southern
    via: American BBQ & Southern

-------------------------------------------------------
top recipes (must contain: ['cumin', 'garlic'])
-------------------------------------------------------
                                                    title  similarity                                                      matched_spices  num_matched
                                          Chicken Karrahi       0.500                         [black peppe

In [ ]:
"""import joblib
import numpy as np

# load your saved model
m = joblib.load("spicerack_model.joblib")

print("keys:", list(m.keys()))
print("recipes:", m["n_recipes"])
print("clusters:", m["n_clusters"])
print("recipe_matrix shape:", m["recipe_matrix"].shape)
print()

# test with a pantry
test_pantry = ["garlic powder", "cumin", "paprika", "salt", "saffron", "oregano"]

# build user vector
user_bin = np.asarray(m["mlb"].transform([set(test_pantry)]))
user_svd = m["svd"].transform(user_bin)[0]
user_svd = user_svd / np.linalg.norm(user_svd)

# find nearest cluster
nearest = int(m["kmeans"].predict(user_bin)[0])
print(f"nearest cluster: {nearest}")
print(f"cluster top spices: {m['cluster_top_spices'][nearest]}")
print()

# score recipes in that cluster
cluster_arr = np.array(m["cluster_labels"])
scores = m["recipe_matrix"] @ user_svd
scores[cluster_arr != nearest] = 0

# show top 5
top_idx = np.argsort(scores)[::-1][:5]
print("top 5 recommendations:")
for i in top_idx:
    if scores[i] > 0:
        matched = sorted(set(test_pantry) & set(m["recipe_spices"][i]))
        print(f"  {scores[i]:.3f}  {m['recipe_titles'][i]}  matched={matched}")
        
# ── missing spices per recommendation ─────────────────────────────────────
print("\ntop 5 with missing spices:")
for i in top_idx:
    if scores[i] > 0:
        recipe_spices = set(m["recipe_spices"][i])
        matched = sorted(set(test_pantry) & recipe_spices)
        missing = sorted(recipe_spices - set(test_pantry))
        print(f"  {scores[i]:.3f}  {m['recipe_titles'][i]}")
        print(f"           have:    {matched}")
        print(f"           missing: {missing}")

# ── recommended spices to buy ─────────────────────────────────────────────
# for every spice you don't have, count how many recipes
# become fully matched if you add just that one spice
from collections import Counter

pantry_set = set(test_pantry)
unlock     = Counter()

for sp_list in m["recipe_spices"]:
    recipe_set = set(sp_list)
    missing    = recipe_set - pantry_set
    if len(missing) == 1:
        unlock[next(iter(missing))] += 1

print("\nspices to buy next (unlocks the most recipes):")
for spice, count in unlock.most_common(10):
    print(f"  {spice:<25} unlocks {count:,} new recipes")"""

### 



In [ ]:
'''import joblib
import numpy as np

# load your saved model
m = joblib.load("spicerack_model.joblib")

print("keys:", list(m.keys()))
print("recipes:", m["n_recipes"])
print("clusters:", m["n_clusters"])
print("recipe_matrix shape:", m["recipe_matrix"].shape)
print()

# test with a pantry
test_pantry = ["garlic powder", "cumin", "paprika", "salt", "saffron", "oregano"]

# build user vector
user_bin = np.asarray(m["mlb"].transform([set(test_pantry)]))
user_svd = m["svd"].transform(user_bin)[0]
user_svd = user_svd / np.linalg.norm(user_svd)

# find nearest cluster
nearest = int(m["kmeans"].predict(user_bin)[0])
print(f"nearest cluster: {nearest}")
print(f"cluster top spices: {m['cluster_top_spices'][nearest]}")
print()

# score recipes in that cluster
cluster_arr = np.array(m["cluster_labels"])
scores = m["recipe_matrix"] @ user_svd
scores[cluster_arr != nearest] = 0

# show top 5
top_idx = np.argsort(scores)[::-1][:5]
print("top 5 recommendations:")
for i in top_idx:
    if scores[i] > 0:
        matched = sorted(set(test_pantry) & set(m["recipe_spices"][i]))
        print(f"  {scores[i]:.3f}  {m['recipe_titles'][i]}  matched={matched}")
        
# ── missing spices per recommendation ─────────────────────────────────────
print("\ntop 5 with missing spices:")
for i in top_idx:
    if scores[i] > 0:
        recipe_spices = set(m["recipe_spices"][i])
        matched = sorted(set(test_pantry) & recipe_spices)
        missing = sorted(recipe_spices - set(test_pantry))
        print(f"  {scores[i]:.3f}  {m['recipe_titles'][i]}")
        print(f"           have:    {matched}")
        print(f"           missing: {missing}")

# ── recommended spices to buy ─────────────────────────────────────────────
# for every spice you don't have, count how many recipes
# become fully matched if you add just that one spice
from collections import Counter

pantry_set = set(test_pantry)
unlock     = Counter()

for sp_list in m["recipe_spices"]:
    recipe_set = set(sp_list)
    missing    = recipe_set - pantry_set
    if len(missing) == 1:
        unlock[next(iter(missing))] += 1

print("\nspices to buy next (unlocks the most recipes):")
for spice, count in unlock.most_common(10):
    print(f"  {spice:<25} unlocks {count:,} new recipes")'''

In [27]:
import joblib
import numpy as np

m = joblib.load("models/spicerack_model.joblib")

# ── step 1: define a test pantry ──────────────────────────────────────────
pantry   = ["garlic", "cumin", "paprika", "salt", "black pepper"]
pantry_set = set(pantry)

# ── step 2: build user vector ─────────────────────────────────────────────
user_bin = np.asarray(m["mlb"].transform([pantry_set]))
user_svd = m["svd"].transform(user_bin)[0]
user_svd = user_svd / np.linalg.norm(user_svd)

print(f"user vector shape: {user_svd.shape}")
print(f"user vector (first 5 values): {user_svd[:5].round(4)}")

# ── step 3: find nearest cluster ──────────────────────────────────────────
nearest = int(m["kmeans"].predict(user_bin)[0])
print(f"\nnearest cluster: {nearest}")
print(f"top spices in that cluster: {m['cluster_top_spices'][nearest]}")

# ── step 4: score every recipe ────────────────────────────────────────────
scores      = m["recipe_matrix"] @ user_svd
cluster_arr = np.array(m["cluster_labels"])
scores[cluster_arr != nearest] = 0

print(f"\nrecipes in cluster: {(cluster_arr == nearest).sum():,}")
print(f"max score: {scores.max():.4f}")
print(f"non-zero scores: {(scores > 0).sum():,}")

# ── step 5: show top 10 ───────────────────────────────────────────────────
top_idx = np.argsort(scores)[::-1][:10]

print(f"\ntop 10 recommendations for pantry {pantry}:")
print(f"{'score':<8} {'title':<45} matched spices")
print("-" * 80)
for i in top_idx:
    if scores[i] <= 0:
        break
    recipe_spices = set(m["recipe_spices"][i])
    matched = sorted(pantry_set & recipe_spices)
    print(f"{scores[i]:.3f}    {m['recipe_titles'][i][:44]:<45} {matched}")

user vector shape: (100,)
user vector (first 5 values): [ 0.6639  0.3138  0.0824 -0.0093  0.1666]

nearest cluster: 23
top spices in that cluster: ['paprika', 'garlic', 'salt', 'black pepper', 'cayenne', 'onion powder', 'oregano', 'thyme']

recipes in cluster: 13,830
max score: 1.0000
non-zero scores: 13,830

top 10 recommendations for pantry ['garlic', 'cumin', 'paprika', 'salt', 'black pepper']:
score    title                                         matched spices
--------------------------------------------------------------------------------
1.000    Bake-Fried Potatoes                           ['black pepper', 'cumin', 'garlic', 'paprika', 'salt']
1.000    Chickaritos                                   ['black pepper', 'cumin', 'garlic', 'paprika', 'salt']
1.000    Lighter Version of Yotam Ottolenghi & Sami T  ['black pepper', 'cumin', 'garlic', 'paprika', 'salt']
1.000    Classic Hummus with Spiced 'n Baked Pita Chi  ['black pepper', 'cumin', 'garlic', 'paprika', 'salt']
1.000   

### Debugging


In [ ]:
# run this in your notebook
spice_cols = list(mlb.classes_)
print('italian seasoning in mlb:', 'italian seasoning' in spice_cols)

if 'italian seasoning' in spice_cols:
    idx = spice_cols.index('italian seasoning')
    print(f'recipes with italian seasoning: {X[:, idx].sum():,}')
    print(f'that is {X[:, idx].mean()*100:.2f}% of recipes')

In [37]:

m = joblib.load("spicerack_model2.joblib")
print(list(m.keys()))
print("n_recipes:", m["n_recipes"])
print("n_clusters:", m["n_clusters"])

['kmeans', 'svd', 'mlb', 'tfidf', 'idf_boost', 'recipe_matrix', 'recipe_titles', 'recipe_spices', 'cluster_labels', 'cluster_top_spices', 'n_clusters', 'n_recipes', 'silhouette']
n_recipes: 2231142
n_clusters: 100


In [12]:
test = pd.read_csv('/Users/daniellarson/Desktop/SpiceRack/SpiceRack-website-main/data/cluster_data.csv')
test_copy = test.copy()
test_1000 = test_copy.sample(n=1000, random_state=42).reset_index(drop=True)



In [10]:
test.shape

(2231142, 10)

In [ ]:
test.columns

Index(['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source',
       'NER', 'ingredients_parsed', 'spices', 'cluster'],
      dtype='object')